# Answer Quality and Model Fallback

Level 5: Senior FDE / Staff-Level Scenarios.

**Problems in this notebook**

- Problem 42: Simple Evaluator Scoring
- Problem 45: Model Fallback Logic
- Problem 46: Hallucination Risk Scoring

**How to use this notebook**

1. Read the problem and say your plan out loud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `02_answer_quality_and_fallback_THEORY.md`.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 42: Simple Evaluator Scoring

**The situation**

A team is shipping a policy Q&A assistant. Before they add fancy evaluation, they want a simple, repeatable score that checks whether an answer mentions the facts it should.

**Your task**

Write `keyword_score(answer, expected_keywords)` that returns the share of expected keywords found in the answer, rounded to 2 decimals. Return `None` when there is nothing to score.

**Example**

`"Refunds are allowed within 30 days."`, `["refunds", "30 days"]` -> `1.0`

`"Refunds are allowed."`, `["refunds", "30 days"]` -> `0.5`

**Ask before coding**

- Exact text match, or should synonyms count?
- Should case and extra spaces matter?
- What should an empty expected list return?

**Plan to say aloud**

1. Clean the expected keywords: lowercase, single spaces, drop blanks and repeats.
2. If nothing is left, return None.
3. Clean the answer the same way.
4. Count keywords found in the answer and divide by the number of keywords.

### Prompt for your AI assistant

```text
Write a Python function keyword_score(answer: str, expected_keywords: list[str]) -> float | None.
Rules: lowercase both sides and collapse runs of whitespace to one space. Ignore blank keywords
and count repeated keywords once. If no keywords are left, return None.
A keyword is a hit if it appears as a substring of the cleaned answer.
Return hits / number of keywords, rounded to 2 decimals. Standard library only.
Show only the function.
```

**Check the AI's code for**

- Returns `1.0` for an empty keyword list, so a broken eval set looks like a perfect score.
- Lowercases the answer but not the keywords, so `"Refunds"` never matches.
- Does not collapse spaces, so `"30   days"` misses `"30 days"`.
- Counts a repeated keyword twice, which skews the score.

In [ ]:
def keyword_score(answer: str, expected_keywords: list[str]) -> float | None:
    """Share of expected keywords found in the answer, rounded to 2 decimals.

    Rules:
    - lowercase both sides, collapse whitespace to single spaces
    - ignore blank keywords, count repeats once
    - no keywords left -> None
    - a hit is a substring match in the cleaned answer
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("Refunds are allowed within 30 days.", ["refunds", "30 days"]), 1.0),
    (("Refunds are allowed within 30   DAYS.", ["Refunds", "30 days"]), 1.0),  # case and spaces
    (("Refunds are allowed.", ["refunds", "30 days"]), 0.5),
    (("No refunds.", ["refund", "receipt", "manager"]), 0.33),  # note: "no refunds" still hits "refund"
    (("Refunds in 30 days.", ["refunds", "refunds", " "]), 1.0),  # repeats and blanks
    (("Anything at all.", []), None),  # nothing to score
]
run_checks(keyword_score, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def keyword_score_reference(answer: str, expected_keywords: list[str]) -> float | None:
    def clean(text: str) -> str:
        return " ".join(text.lower().split())

    keywords = {clean(k) for k in expected_keywords if k.strip()}
    if not keywords:
        return None
    text = clean(answer)
    hits = sum(1 for k in keywords if k in text)
    return round(hits / len(keywords), 2)


assert run_checks(keyword_score_reference, cases)

**Explain it in 30 seconds**

> "This is a baseline, a simple score I can explain and repeat. I clean both sides the same way, count how many expected keywords show up, and divide. An empty list returns None, so a broken test set never looks like a pass. It cannot catch contradictions: "No refunds" still matches "refund", so I would never treat it as the truth."

**Edge cases to mention**

- Empty or blank keyword list.
- Negation: "refunds are not allowed" still scores a hit.
- Partial words: "refund" matches inside "refundable".
- Synonyms: "one month" does not match "30 days".

**If they ask "how would this work in production?"**

- Keep a human-labeled eval set and track the score per version of the prompt and model.
- Add citation checks and a careful model-graded judge for things keywords cannot see.
- Look at the cases where the keyword score and human reviewers disagree.

## Problem 45: Model Fallback Logic

**The situation**

A customer support assistant calls a primary model, then backup models in order if the primary fails or times out. But if the primary refused for safety reasons, trying another model just to get an answer would bypass the safety rule.

**Your task**

Write `choose_fallback_response(responses)` that walks the responses in priority order and returns the first good text. A safety block stops the search and returns `None`.

**Example**

`[{"ok": False, "error": "timeout"}, {"ok": True, "text": "answer"}]` -> `"answer"`

`[{"ok": False, "error": "safety_block"}, {"ok": True, "text": "answer"}]` -> `None`

**Ask before coding**

- Which errors are safe to fall back on, like timeouts and rate limits?
- Should a safety refusal ever be retried on another model?
- Should the user be told that a backup model answered?

**Plan to say aloud**

1. Go through the responses in priority order.
2. If a response is a safety block, stop and return None.
3. If a response is ok and has non-blank text, return that text.
4. If nothing worked, return None.

### Prompt for your AI assistant

```text
Write a Python function choose_fallback_response(responses: list[dict]) -> str | None.
Rules: responses are in priority order. Each has "ok" (bool), maybe "text" and maybe "error".
If a response has error == "safety_block", stop and return None. Do not look further.
Return the text of the first response where ok is True and text is not blank.
If none qualify, return None. Show only the function.
```

**Check the AI's code for**

- Skips past a safety block to the next model. That is exactly the bypass we must prevent.
- Returns whitespace-only text as a valid answer.
- Treats a truthy string like `"false"` in `ok` as success; check for `True`.
- Raises `KeyError` on a response with no `text` field.

In [ ]:
def choose_fallback_response(responses: list[dict]) -> str | None:
    """Return the first good text in priority order.

    Rules:
    - error == "safety_block" -> stop and return None (never bypass safety)
    - ok is True and text is not blank -> return text
    - nothing qualifies -> None
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"model": "primary", "ok": False, "error": "timeout"},
       {"model": "backup", "ok": True, "text": "answer"}],), "answer"),
    (([{"model": "primary", "ok": True, "text": "first"},
       {"model": "backup", "ok": True, "text": "second"}],), "first"),
    (([{"model": "primary", "ok": False, "error": "safety_block"},
       {"model": "backup", "ok": True, "text": "unsafe answer"}],), None),  # never bypass safety
    (([{"model": "primary", "ok": True, "text": "   "},
       {"model": "backup", "ok": True, "text": "real"}],), "real"),        # blank text is not an answer
    (([],), None),
]
run_checks(choose_fallback_response, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def choose_fallback_response_reference(responses: list[dict]) -> str | None:
    for response in responses:
        if response.get("error") == "safety_block":
            return None
        text = response.get("text") or ""
        if response.get("ok") is True and text.strip():
            return text
    return None


assert run_checks(choose_fallback_response_reference, cases)

**Explain it in 30 seconds**

> "Fallback keeps the product up when a model times out, but it must not become a way around safety. I walk the models in priority order and take the first real answer. If any model refused for safety, I stop right there and return nothing. In production I would also track how often we fall back, because a backup model may give weaker answers."

**Edge cases to mention**

- Every model fails.
- A model returns ok but empty or blank text.
- A safety block from the second model after the first one timed out.
- A tenant that is not allowed to use a certain backup model.

**If they ask "how would this work in production?"**

- Only fall back on errors that are temporary, like timeouts or rate limits, and cap retries so we do not make an outage worse.
- Record which model answered and why, and alert when the fallback rate jumps.
- Respect per-tenant rules about which models may see their data.

## Problem 46: Hallucination Risk Scoring

**The situation**

A RAG assistant for an insurance company answers questions from policy documents. Some answers have weak retrieval support or cite documents that were never retrieved. Those answers should be flagged before a customer sees them.

**Your task**

Write `hallucination_risk(citations, retrieved_ids, top_score)` that returns `"low"`, `"medium"` or `"high"`. A citation only counts if it points to a document that was actually retrieved.

**Example**

`citations=[]`, `retrieved_ids=["d1"]`, `top_score=0.9` -> `"high"`

`citations=["d1"]`, `retrieved_ids=["d1"]`, `top_score=0.4` -> `"medium"`

**Ask before coding**

- Are citations checked against what we retrieved, or just taken from the model?
- What retrieval scores count as strong and weak?
- Should high risk block the answer or send it to a reviewer?

**Plan to say aloud**

1. No citations, or any citation that was not retrieved, means high.
2. A top score under 0.3 means high.
3. A top score under 0.6 means medium.
4. Otherwise low.

### Prompt for your AI assistant

```text
Write a Python function hallucination_risk(citations: list[str], retrieved_ids: list[str],
top_score: float) -> str.
Rules: return "high" if citations is empty or any citation is not in retrieved_ids.
Then return "high" if top_score < 0.3, "medium" if top_score < 0.6, else "low".
Show only the function.
```

**Check the AI's code for**

- Counts citations without checking them against `retrieved_ids`, so a made-up source lowers the risk.
- Uses `<=` instead of `<`, so a score of exactly 0.6 becomes medium.
- Checks the score before the citations, so a strong score hides a missing citation.

In [ ]:
def hallucination_risk(citations: list[str], retrieved_ids: list[str], top_score: float) -> str:
    """Return "low", "medium" or "high".

    Rules:
    - no citations, or any citation not in retrieved_ids -> "high"
    - top_score < 0.3 -> "high"
    - top_score < 0.6 -> "medium"
    - otherwise -> "low"
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([], ["d1"], 0.9), "high"),            # no citations
    ((["d1"], ["d1", "d2"], 0.7), "low"),
    ((["d1"], ["d1"], 0.4), "medium"),
    ((["d9"], ["d1"], 0.9), "high"),        # cites a document we never retrieved
    ((["d1"], ["d1"], 0.2), "high"),        # weak retrieval
    ((["d1"], ["d1"], 0.6), "low"),         # boundary
]
run_checks(hallucination_risk, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def hallucination_risk_reference(citations: list[str], retrieved_ids: list[str], top_score: float) -> str:
    retrieved = set(retrieved_ids)
    if not citations or any(c not in retrieved for c in citations):
        return "high"
    if top_score < 0.3:
        return "high"
    if top_score < 0.6:
        return "medium"
    return "low"


assert run_checks(hallucination_risk_reference, cases)

**Explain it in 30 seconds**

> "A hallucination is when the model states something its sources do not support. This is a simple warning light, not a truth check. No citations, a citation to a document we never retrieved, or weak retrieval all mean high risk. Citations only lower the risk if they point to context we actually gave the model."

**Edge cases to mention**

- A real citation that does not support the claim it is attached to.
- A correct answer with a low retrieval score.
- Numbers in the answer that appear nowhere in the sources.
- No documents retrieved at all.

**If they ask "how would this work in production?"**

- Split the answer into claims and check each claim against the cited text.
- Use stricter thresholds in regulated domains like health or finance.
- Let the assistant say "I don't know" or send high-risk answers to a human reviewer.

## Wrap-up

- Simple scores are useful because you can explain them, but none of them prove an answer is true.
- Never let a quality or uptime trick, like fallback, get around a safety rule.
- Check model output against what you actually gave it: retrieved ids, expected facts, real text.